In [ ]:
import pandas as pd
import yfinance as yf
from google.cloud import bigquery

PROJECT = "fluid-terminal-465516-s7"
DATASET = "magic_formula"

MISSING_TBL = f"{PROJECT}.{DATASET}.bt_missing_sell_windows_d3"
MAP_TBL     = f"{PROJECT}.{DATASET}.symbol_yf_map"
OUT_TBL     = f"{PROJECT}.{DATASET}.daily_price_yf_d3_sell"

bq = bigquery.Client(project=PROJECT)

missing = bq.query(f"""
SELECT m.symbol, map.yf_ticker, m.need_start, m.need_end
FROM `{MISSING_TBL}` m
JOIN `{MAP_TBL}` map USING(symbol)
""").to_dataframe()

missing["need_start"] = pd.to_datetime(missing["need_start"])
missing["need_end"]   = pd.to_datetime(missing["need_end"])

# Collapse per ticker: min start, max end (reduces API calls)
ranges = (missing.groupby(["symbol","yf_ticker"], as_index=False)
          .agg(start=("need_start","min"), end=("need_end","max")))

all_rows = []
failed = []

for _, r in ranges.iterrows():
    sym = str(r["symbol"])
    тик = str(r["yf_ticker"])
    start = r["start"].date()
    end   = (r["end"] + pd.Timedelta(days=1)).date()  # yfinance end is exclusive

    try:
        hist = yf.Ticker(тик).history(start=start, end=end, auto_adjust=False)
        if hist is None or hist.empty:
            failed.append((sym, тик, "empty_history"))
            continue

        df = hist.reset_index()

        # normalize date column
        if "Date" in df.columns:
            df.rename(columns={"Date": "date"}, inplace=True)
        elif "Datetime" in df.columns:
            df.rename(columns={"Datetime": "date"}, inplace=True)

        df.rename(columns={
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume"
        }, inplace=True)

        if "adj_close" not in df.columns:
            df["adj_close"] = df.get("close")

        df["symbol"] = sym
        df["source"] = "yfinance"

        keep = ["symbol","date","adj_close","close","open","high","low","volume","source"]
        for c in keep:
            if c not in df.columns:
                df[c] = None
        df = df[keep]

        # type coercions for BigQuery/pyarrow
        df["symbol"] = df["symbol"].astype(str)
        df["source"] = df["source"].astype(str)
        df["date"] = pd.to_datetime(df["date"]).dt.date

        for c in ["adj_close","close","open","high","low"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        df["volume"] = pd.to_numeric(df["volume"], errors="coerce").fillna(0).astype("int64")

        df = df[df["adj_close"].notna()]
        if df.empty:
            failed.append((sym, тик, "all_nan_prices"))
            continue

        all_rows.append(df)

    except Exception as e:
        failed.append((sym, тик, f"exception:{type(e).__name__}"))
        continue

if not all_rows:
    print("No rows fetched. Failed sample:", failed[:20])
else:
    out = pd.concat(all_rows, ignore_index=True)

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND",
        schema=[
            bigquery.SchemaField("symbol", "STRING"),
            bigquery.SchemaField("date", "DATE"),
            bigquery.SchemaField("adj_close", "FLOAT"),
            bigquery.SchemaField("close", "FLOAT"),
            bigquery.SchemaField("open", "FLOAT"),
            bigquery.SchemaField("high", "FLOAT"),
            bigquery.SchemaField("low", "FLOAT"),
            bigquery.SchemaField("volume", "INTEGER"),
            bigquery.SchemaField("source", "STRING"),
        ],
    )

    job = bq.load_table_from_dataframe(out, OUT_TBL, job_config=job_config)
    job.result()

    print("Loaded rows:", len(out))
    print("Failed tickers:", len(failed))
    print("Failed sample:", failed[:20])